# 123 Critique: train curriculum + robust mod_taylor and compare

This notebook runs:
1) training of two mod_taylor variants (optional): baseline_curriculum and robust_curriculum;
2) evaluation on critique scenarios with the same metrics as notebooks 120/121/122;
3) display of all generated CSV/PNG outputs.

In [ ]:
import os
import sys
import shlex
import json
import pathlib
import subprocess

import pandas as pd
import torch
from IPython.display import Image, Markdown, display


def _find_project_root() -> pathlib.Path:
    here = pathlib.Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "src").is_dir() and (p / "scripts" / "train_mod_taylor_critique_variants.py").is_file():
            return p
    cand = pathlib.Path("/content/econml")
    if (cand / "src").is_dir() and (cand / "scripts" / "train_mod_taylor_critique_variants.py").is_file():
        return cand
    raise RuntimeError("Project root not found.")


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_ROOT = str(PROJECT_ROOT / "artifacts")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "float32"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)

In [ ]:
DO_TRAIN = False  # set True to actually train both variants
RUNS_JSON = PROJECT_ROOT / "artifacts" / "critique_mod_taylor_variants" / "latest_runs.json"

train_cmd = [
    sys.executable,
    "scripts/train_mod_taylor_critique_variants.py",
    "--artifacts-root", ARTIFACTS_ROOT,
    "--device", DEVICE,
    "--dtype", DTYPE,
    "--save-runs-json", str(RUNS_JSON),
]

if DO_TRAIN:
    print("Running:", " ".join(shlex.quote(x) for x in train_cmd))
    subprocess.run(train_cmd, cwd=str(PROJECT_ROOT), check=True)
else:
    print("Training skipped (DO_TRAIN=False). Using existing runs_json if present:", RUNS_JSON)

In [ ]:
runs = {}
if RUNS_JSON.exists():
    runs = json.loads(RUNS_JSON.read_text(encoding="utf-8"))

baseline_run_dir = runs.get("baseline_curriculum", {}).get("run_dir", "")
robust_run_dir = runs.get("robust_curriculum", {}).get("run_dir", "")

FIG_DIR = PROJECT_ROOT / "artifacts" / "critique_mod_taylor_variants_nb123"
os.makedirs(FIG_DIR, exist_ok=True)

cmd = [
    sys.executable,
    "scripts/build_critique_mod_taylor_variants.py",
    "--artifacts-root", ARTIFACTS_ROOT,
    "--fig-dir", str(FIG_DIR),
    "--device", DEVICE,
    "--dtype", "float64",
    "--use-selected", "true",
    "--cons-mode", "author",
    "--ensure-postprocess", "true",
    "--ensure-ir", "true",
    "--baseline-label", "baseline_curriculum",
    "--robust-label", "robust_curriculum",
]
if baseline_run_dir:
    cmd += ["--baseline-run-dir", str(baseline_run_dir)]
if robust_run_dir:
    cmd += ["--robust-run-dir", str(robust_run_dir)]

print("Running:", " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)
print("Done. FIG_DIR=", FIG_DIR)

In [ ]:
pngs = sorted(FIG_DIR.glob("*.png"))
for p in pngs:
    display(Markdown(f"### {p.name}"))
    display(Image(filename=str(p), width=980))

for name in [
    "critique_shock_train_summary.csv",
    "critique_shock_train_delta.csv",
    "compare_shock_train_vs_baseline.csv",
    "critique_bad_uncertainty_moments.csv",
    "compare_bad_uncertainty_bad_regime_vs_baseline.csv",
    "critique_combined_summary.csv",
    "critique_combined_delta.csv",
    "compare_combined_vs_baseline.csv",
]:
    p = FIG_DIR / name
    if p.exists():
        display(Markdown(f"### {name}"))
        display(pd.read_csv(p))
    else:
        print("Missing:", p)